In [1]:
from util import import_ragas_custom, load_env_variables_from_all_env_files

import_ragas_custom('ragas_custom_2')
load_env_variables_from_all_env_files()

Arquivos copiados com sucesso!


In [2]:
import os
import asyncio
import nest_asyncio

import pandas as pd
import numpy as np

from ragas.integrations.llama_index import evaluate
from ragas.run_config import RunConfig
from ragas.testset.synthesizers.testset_schema import Testset
from llama_index.llms.openai import OpenAI, AsyncOpenAI
from llama_index.embeddings.openai import OpenAIEmbedding, OpenAIEmbeddingModelType
from ragas.run_config import RunConfig

from llama_index.core import (
    VectorStoreIndex,
    SimpleDirectoryReader,
    Settings,
)

from ragas.metrics import (
    faithfulness,
    answer_relevancy,
    context_precision,
    context_recall,
)

from ragas.metrics._aspect_critic import harmfulness

[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\jmess\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
c:\Users\jmess\miniconda3\envs\rag_test\Lib\site-packages\ragas\prompt\base.py:9: LangChainDeprecationWarning: As of langchain-core 0.3.0, LangChain uses pydantic v2 internally. The langchain_core.pydantic_v1 module was a compatibility shim for pydantic v1, and should no longer be used. Please update the code to import from Pydantic directly.

For example, replace imports like: `from langchain_core.pydantic_v1 import BaseModel`
with: `from pydantic import BaseModel`
or the v1 compatibility namespace if you are working in a code base that has not been fully upgraded to pydantic 2 yet. 	from pydantic.v1 import BaseModel

  from ragas.llms.prompt import PromptValue


In [3]:
nest_asyncio.apply()

In [4]:
DATA_PATH = 'data'
TESTSET = 'testset_openai_4omini.jsonl'
PERSIST_DIR = "./storage"
LANGUAGE = 'portuguese'
TIMEOUT = 2400
RESULT_CSV = 'result__gpt4mini_gpt4mini.csv'
MODEL = 'gpt-4o-mini-2024-07-18'
CACHE_DIR = 'cache_2'

In [5]:
run_config = RunConfig(max_workers=2, timeout=TIMEOUT)

In [6]:
testset = Testset.from_jsonl(TESTSET).to_pandas()

data = pd.DataFrame(testset['eval_sample'].to_list())

print("Tamanho do dataset: ", len(data))
print(data.head())

Tamanho do dataset:  7
                                          user_input retrieved_contexts  \
0  Qual é a importância de um tema na construção ...               None   
1  importância do contexto na formulação de pergu...               None   
2  Como as soluções de No-Break da CM Comandos se...               None   
3  Como as abordagens de suporte tecnico e inovac...               None   
4  compromissos CM Comandos qualidade produtos pr...               None   

                                  reference_contexts response multi_responses  \
0                                                 []     None            None   
1                                                 []     None            None   
2  [A CM Comandos, com mais de 38 anos de atuação...     None            None   
3  [A CM Comandos, com mais de 38 anos de atuação...     None            None   
4  [A EMPRESACom mais de 38 anos de atuação, a CM...     None            None   

                                       

In [7]:
non_null_columns = data.loc[:, data.notna().any(axis=0)]
nan_rows = non_null_columns[non_null_columns.isna().any(axis=1)]

print("Quantidade de nulos: ", len(nan_rows))
print(nan_rows)

del nan_rows

Quantidade de nulos:  0
Empty DataFrame
Columns: [user_input, reference_contexts, reference]
Index: []


In [8]:
cleaned_data = non_null_columns.dropna()
null_columns = data.loc[:, data.isna().all(axis=0)]
data = pd.concat([cleaned_data, null_columns], axis=1)

In [9]:
print("Quantidade de linhas após remoção de nulos: ", len(data))
print(data)

Quantidade de linhas após remoção de nulos:  7
                                          user_input  \
0  Qual é a importância de um tema na construção ...   
1  importância do contexto na formulação de pergu...   
2  Como as soluções de No-Break da CM Comandos se...   
3  Como as abordagens de suporte tecnico e inovac...   
4  compromissos CM Comandos qualidade produtos pr...   
5  Quais os beneficio da maior precisao dos equip...   
6  Quais sã as prinpais caracteristicas dos No Br...   

                                  reference_contexts  \
0                                                 []   
1                                                 []   
2  [A CM Comandos, com mais de 38 anos de atuação...   
3  [A CM Comandos, com mais de 38 anos de atuação...   
4  [A EMPRESACom mais de 38 anos de atuação, a CM...   
5  [APLICAÇÕES\nOs equipamentos da CM Comandos op...   
6  [Ao pensar em adquirir No Breaks,     \nconte ...   

                                           reference re

In [10]:
testset['eval_sample'] = data.to_dict(orient='records')

In [11]:
testset = Testset.from_pandas(testset)

In [12]:
embeding = OpenAIEmbedding(model=OpenAIEmbeddingModelType.TEXT_EMBED_3_SMALL)
model = OpenAI(model=MODEL)

Settings.embed_model = embeding
Settings.llm = model

In [13]:
metrics = [
    faithfulness,
    answer_relevancy,
    context_precision,
    context_recall,
    harmfulness,
]

for query in metrics:
    path = os.path.join(CACHE_DIR, query.__class__.__name__)
    if not os.path.exists(path):
        os.makedirs(path)

    try:
        prompts = query.load_prompts(path, LANGUAGE)
        query.set_prompts(**prompts)
    except Exception:
        prompts = asyncio.run(query.adapt_prompts(LANGUAGE, None))
        query.set_prompts(**prompts)
        query.save_prompts(path)
        prompts = query.load_prompts(path, LANGUAGE)
        query.set_prompts(**prompts)


In [14]:
documents = SimpleDirectoryReader(DATA_PATH).load_data()
index = VectorStoreIndex.from_documents(documents, show_progress=True)
query_engine = index.as_query_engine(request_timeout=TIMEOUT)

Parsing nodes:   0%|          | 0/58 [00:00<?, ?it/s]

Generating embeddings:   0%|          | 0/82 [00:00<?, ?it/s]

In [15]:
result = evaluate(
    query_engine=query_engine,
    metrics=metrics,
    dataset=testset,
    llm=model,
    embeddings=embeding,
    run_config=run_config
)

Running Query Engine:   0%|          | 0/7 [00:00<?, ?it/s]

Dataset({
    features: ['user_input', 'retrieved_contexts', 'response', 'reference'],
    num_rows: 7
})


Evaluating:   0%|          | 0/35 [00:00<?, ?it/s]

In [16]:
result_dataframe = result.to_pandas()
result_dataframe.to_csv(RESULT_CSV)

In [17]:
print(result)

{'faithfulness': 0.6187, 'answer_relevancy': 0.7533, 'context_precision': 0.4286, 'context_recall': 0.1905, 'harmfulness': 0.5714}
